# Analysis: Positional Encoding Comparison

This notebook reproduces all figures and tables for the paper.

**Dependencies**: torch, matplotlib, numpy, pandas, seaborn

In [ ]:
import json
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 150
plt.rcParams['font.size'] = 12

RESULTS_DIR = Path('../results')
CHECKPOINTS_DIR = Path('../checkpoints')

## 1. Load Results

In [ ]:
# Load stored results (JSON format from evaluate.py)
methods = ['learned', 'sinusoidal', 'rope', 'alibi', 'nope', 'kerple', 'cable', 'position_interpolation']
results = {}

for method in methods:
    path = RESULTS_DIR / f'{method}_results.json'
    if path.exists():
        with open(path) as f:
            results[method] = json.load(f)
        print(f'Loaded {method}')
    else:
        print(f'Missing {method}')

if not results:
    print('WARNING: No results found. Run experiments first.')
    # Use placeholder data for testing
    results = {m: {'ppl': {512: 25, 1024: 40, 2048: 80, 4096: 200, 8192: 500}} for m in methods}

## 2. Figure 1: PPL vs Sequence Length

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
colors = sns.color_palette('husl', len(methods))

lengths = [512, 1024, 2048, 4096, 8192]
markers = ['o', 's', 'D', '^', 'v', '<', '>', 'p']

for i, (method, data) in enumerate(results.items()):
    if 'ppl' not in data:
        continue
    ppl_values = [data['ppl'].get(str(L), np.nan) for L in lengths]
    ax.plot(lengths, ppl_values, marker=markers[i], label=method.upper(), 
            color=colors[i], linewidth=2, markersize=6)

ax.set_xscale('log', base=2)
ax.set_yscale('log')
ax.set_xlabel('Sequence Length')
ax.set_ylabel('Perplexity')
ax.set_title('Language Modeling Perplexity vs Sequence Length')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
ax.axvline(x=512, color='gray', linestyle='--', alpha=0.5, label='Training length')
plt.tight_layout()
plt.savefig('fig1_ppl_vs_length.pdf', bbox_inches='tight')
plt.show()

## 3. Figure 2: Needle-in-Haystack Accuracy

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Accuracy at fixed depth (75%) across lengths
ax = axes[0]
for i, (method, data) in enumerate(results.items()):
    if 'needle' not in data:
        continue
    accs = [data['needle'].get(str(L), {}).get('0.75', np.nan) for L in lengths]
    ax.plot(lengths, accs, marker=markers[i], label=method.upper(), 
            color=colors[i], linewidth=2)
ax.set_xlabel('Sequence Length')
ax.set_ylabel('Accuracy')
ax.set_title('Needle Retrieval at 75% Context Depth')
ax.legend()

# Right: Accuracy across depths at L=2048
ax = axes[1]
depths = [0.25, 0.50, 0.75, 0.90]
for i, (method, data) in enumerate(results.items()):
    if 'needle' not in data:
        continue
    accs = [data['needle'].get('2048', {}).get(str(d), np.nan) for d in depths]
    ax.plot(depths, accs, marker=markers[i], label=method.upper(), 
            color=colors[i], linewidth=2)
ax.set_xlabel('Context Depth')
ax.set_ylabel('Accuracy')
ax.set_title('Needle Retrieval vs Context Depth (L=2048)')
ax.legend()

plt.tight_layout()
plt.savefig('fig2_needle_accuracy.pdf', bbox_inches='tight')
plt.show()

## 4. Figure 3: Recency Bias Score

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

bias_scores = []
method_names = []
for method, data in results.items():
    if 'recency' in data and 'bias_score' in data['recency']:
        bias_scores.append(data['recency']['bias_score'])
        method_names.append(method.upper())

bars = ax.bar(range(len(method_names)), bias_scores, color=colors[:len(method_names)])
ax.set_xticks(range(len(method_names)))
ax.set_xticklabels(method_names, rotation=45)
ax.set_ylabel('Recency Bias Score (near acc - far acc)')
ax.set_title('Recency Bias by Positional Encoding Method')
ax.axhline(y=0, color='black', linewidth=0.5)

# Add value labels
for bar, score in zip(bars, bias_scores):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(), 
            f'{score:.3f}', ha='center', va='bottom' if score >= 0 else 'top')

plt.tight_layout()
plt.savefig('fig3_recency_bias.pdf', bbox_inches='tight')
plt.show()

## 5. Figure 4: Compute vs Extrapolation

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

flops = []
ppl_ratios = []
method_labels = []

for method, data in results.items():
    if 'profile' not in data or 'ppl' not in data:
        continue
    flops.append(data['profile']['flops'] / 1e9)  # in GFLOPs
    ppl_512 = data['ppl'].get('512', 1)
    ppl_8192 = data['ppl'].get('8192', float('inf'))
    ratio = ppl_8192 / ppl_512 if ppl_512 > 0 else float('inf')
    ppl_ratios.append(ratio)
    method_labels.append(method.upper())

scatter = ax.scatter(flops, ppl_ratios, c=range(len(method_labels)), 
                     cmap='viridis', s=200, alpha=0.8)
for i, label in enumerate(method_labels):
    ax.annotate(label, (flops[i], ppl_ratios[i]), 
                xytext=(5, 5), textcoords='offset points', fontsize=8)

ax.set_xlabel('FLOPs (GFLOPs)')
ax.set_ylabel('PPL Ratio (8192 / 512)')
ax.set_title('Compute Cost vs Extrapolation Quality')
ax.set_yscale('log')
plt.tight_layout()
plt.savefig('fig4_compute_vs_extrapolation.pdf', bbox_inches='tight')
plt.show()

## 6. Table 1: Experimental Comparison

In [ ]:
rows = []
for method, data in results.items():
    if 'ppl' not in data:
        continue
    row = {'Method': method.upper()}
    for L in [512, 1024, 2048, 4096, 8192]:
        ppl = data['ppl'].get(str(L), float('nan'))
        row[f'L={L}'] = f'{ppl:.2f}' if not math.isnan(ppl) else '-'
    ppl_512 = data['ppl'].get('512', 1)
    ppl_8192 = data['ppl'].get('8192', float('nan'))
    ratio = ppl_8192 / ppl_512 if (ppl_512 > 0 and not math.isnan(ppl_8192)) else float('nan')
    row['Ratio'] = f'{ratio:.2f}' if not math.isnan(ratio) else '-'
    rows.append(row)

df = pd.DataFrame(rows)
display(df)
df.to_csv('table1_experimental_ppl.csv', index=False)

## 7. Figure 5: Attention Map Comparison

In [ ]:
@torch.no_grad()
def get_attention_maps(model, input_ids, head_idx=0):
    """Extract attention probabilities from all layers."""
    model.eval()
    attention_maps = []
    
    # Register hooks or use forward to get attn
    def hook_fn(module, input, output):
        attention_maps.append(output.cpu())
    
    handles = []
    for block in model.blocks:
        handle = block.attn.register_forward_hook(hook_fn)
        handles.append(handle)
    
    _ = model(input_ids)
    
    for h in handles:
        h.remove()
    
    return attention_maps

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
methods_for_map = ['rope', 'alibi', 'kerple', 'cable']

for idx, method in enumerate(methods_for_map):
    if method not in results:
        continue
    # Load model and generate attention maps
    # This is a placeholder: run with actual model checkpoint
    ax = axes[0, idx]
    ax.text(0.5, 0.5, f'{method.upper()}\n(load checkpoint)', 
            ha='center', va='center', transform=ax.transAxes)
    ax.set_title(f'{method.upper()} - Layer 0')

plt.tight_layout()
plt.savefig('fig5_attention_maps.pdf', bbox_inches='tight')
plt.show()

## 8. Summary Statistics

In [ ]:
print("=== Summary ===")
for method, data in results.items():
    ppl_512 = data.get('ppl', {}).get('512', float('nan'))
    ppl_8192 = data.get('ppl', {}).get('8192', float('nan'))
    ratio = ppl_8192 / ppl_512 if (ppl_512 > 0 and not math.isnan(ppl_8192)) else float('nan')
    needle = data.get('needle', {}).get('2048', {}).get('0.75', float('nan'))
    bias = data.get('recency', {}).get('bias_score', float('nan'))
    flops = data.get('profile', {}).get('flops', float('nan'))
    
    print(f"{method.upper():25s} | PPL@512={ppl_512:7.2f} | Ratio={ratio:7.2f} | "
          f"Needle@75%={needle:6.3f} | Bias={bias:+7.3f} | FLOPs={flops:12,.0f}")